In [1]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb

In [4]:
DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_5c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()


In [5]:
target = df.groupby(["customer_id", "product_id"])["tn"].shift(-2)
# obtengo las 20 columnas con mayor correlación con el target
correlation = df[numeric_cols].corrwith(target).abs().sort_values(ascending=False)
top_20_cols = correlation.head(20).index.tolist()
print("Top 20 columns with highest correlation to target:")
print(top_20_cols)

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Top 20 columns with highest correlation to target:
['tn_wavelet_0_mean_lag_11', 'tn_wavelet_0_mean_lag_15', 'tn_wavelet_0_mean_lag_8', 'tn_rolling_mean_12', 'tn_wavelet_0_mean_lag_20', 'tn_wavelet_0_mean_lag_1', 'tn_wavelet_0_mean', 'tn_wavelet_0_mean_lag_2', 'tn_rolling_mean_12_lag_1', 'tn_wavelet_0_mean_lag_3', 'tn_wavelet_0_mean_lag_6', 'tn_rolling_mean_24', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_24_lag_1', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_24_lag_2', 'tn_rolling_mean_24_lag_3', 'tn_rolling_mean_6', 'tn_rolling_mean_6_lag_8']


In [6]:
# creo el target
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)

kaggle_df = df[df["date_id"] == df["date_id"].max()]
# elimino las rows donde target es nan
df = df[df["target"].notna()]
# hago el scaling del target por tn_std


/tmp/ipykernel_440610/924281194.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [7]:
df["target"].describe()

count    174456.000000
mean          7.122980
std          34.977936
min           0.000000
25%           0.022410
50%           0.374870
75%           2.392753
max        1458.883179
Name: target, dtype: float64

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import BaseCrossValidator

class CustomTimeSeriesSplitter(BaseCrossValidator):
    def __init__(self, subsample_prop=0.5, test_months=1, random_state=None):
        assert 0 < subsample_prop <= 1, "subsample_prop must be in (0, 1]"
        self.subsample_prop = subsample_prop
        self.test_months = test_months
        self.random_state = np.random.RandomState(random_state)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.test_months

    def split(self, X, y=None, groups=None):
        df = X.reset_index(drop=True).copy()
        max_date = df['date_id'].max()
        pair_col = ['product_id', 'customer_id']

        # Calcular tn total y asignar deciles
        total_tn = (
            df.groupby(pair_col)['tn'].sum()
            .reset_index(name='total_tn')
            .sort_values('total_tn', ascending=False)
            .reset_index(drop=True)
        )
        total_tn['quantile'] = pd.qcut(total_tn.index, 10, labels=False)

        # Samplear series por quantil
        sampled_series = []
        for q in range(10):
            group = total_tn[total_tn['quantile'] == q]
            n = max(1, int(len(group) * self.subsample_prop))
            sampled = group.sample(n=n, random_state=self.random_state)
            sampled_series.append(sampled)

        sampled_series_df = pd.concat(sampled_series, ignore_index=True)
        
        # OPTIMIZACIÓN: Usar merge en lugar de apply
        # Marcar series seleccionadas usando merge (mucho más rápido)
        df_marked = df.merge(
            sampled_series_df[pair_col].assign(_selected=True),
            on=pair_col,
            how='left'
        )
        df_marked['_selected'] = df_marked['_selected'].fillna(False)

        for i in range(self.test_months):
            test_date = max_date - i
            
            # Test: todas las series en test_date
            test_idx = np.where(df_marked['date_id'] == test_date)[0].tolist()

            # Train: solo series seleccionadas Y fechas anteriores
            train_idx = np.where(
                (df_marked['_selected'] == True) & 
                (df_marked['date_id'] < test_date-1)
            )[0].tolist()

            yield train_idx, test_idx

            
class SimpleLastDateSplitter(BaseCrossValidator):
    """Split: test = date_id máximo, train = resto. Sin copias innecesarias."""
    def get_n_splits(self, X=None, y=None, groups=None):
        return 1

    def split(self, X, y=None, groups=None):
        # No copies, solo uso la referencia
        max_date = X['date_id'].max()
        test_mask = X['date_id'] == max_date
        
        # Obtener posiciones enteras (para .iloc) en lugar de índices del DataFrame
        test_idx = np.where(test_mask)[0]
        train_idx = np.where(~test_mask)[0]
        
        yield train_idx, test_idx

In [9]:
# Reemplazo de inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        df[col] = df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        df.drop(columns=[col], inplace=True)

# Crear splitter con 20% y 2 meses

# Columnas a dropear
drop_cols = ["fecha", "target", "date_id"]


class CustomMetric:
    def __init__(self, test_df, product_ids):
        # Merge tn_std a test_df para alinear y guardar el resultado
        df_eval = test_df[["product_id", "customer_id", "target"]].copy()
        self.df_eval = df_eval
        self.product_ids = set(product_ids)

    def __call__(self, predt, dtrain):
        df_eval = self.df_eval.copy()
        df_eval["predictions"] = predt
        mask = df_eval["product_id"].isin(self.product_ids)
        df_grouped = df_eval.loc[mask].groupby("product_id", as_index=False)[["predictions", "target"]].sum()
        total_error = np.sum(np.abs(df_grouped["predictions"] - df_grouped["target"])) / np.sum(df_grouped["target"])
        return "total_error", total_error

In [ ]:
import optuna
import numpy as np
import xgboost as xgb
splitter = SimpleLastDateSplitter()
#splitter = CustomTimeSeriesSplitter(subsample_prop=0.1, test_months=2, random_state=42)
def objective(trial):
    total_errors = []
    num_iterations_list = []

    # Hiperparámetros a optimizar
    params = {
        'objective': 'reg:tweedie',
        "device": "cuda",  # Usar GPU
        'tree_method': 'hist',
        'verbosity': 0,
        'sampling_method': 'uniform',  # Aproximación de extra_trees
        "max_depth": 0,
        'tweedie_variance': trial.suggest_float("tweedie_variance", 1.1, 1.9),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1),
        'num_leaves': trial.suggest_int("num_leaves", 8, 256),
        'lambda': trial.suggest_float("lambda", 0.0, 10.0),
        'alpha': trial.suggest_float("alpha", 0.0, 10.0),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 100),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.3, 1.0),
    }

    for fold, (train_idx, test_idx) in enumerate(splitter.split(df)):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        train_df = train_df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)

        print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}, Fold: {fold + 1}")

        X_train = train_df.drop(columns=[col for col in drop_cols if col in train_df.columns])
        y_train = train_df["target"]
        w_train = np.log1p(train_df["tn"]).clip(lower=1)

        X_test = test_df.drop(columns=[col for col in drop_cols if col in test_df.columns])
        y_test = test_df["target"]

        dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

        custom_metric = CustomMetric(
            test_df=test_df,
            product_ids=product_ids
        )
        evals_result = {}
        model = xgb.train(
            params,
            dtrain=dtrain,
            num_boost_round=9999,
            evals=[(dtest, "eval")],
            early_stopping_rounds=int(400 + 4 / params["learning_rate"]),
            custom_metric=custom_metric, 
            evals_result=evals_result,
            verbose_eval=100,
        )
        # get best iteration total error
        total_error = min(evals_result["eval"]["total_error"])

        total_errors.append(total_error)
        num_iterations_list.append(model.best_iteration)

    avg_error = np.mean(total_errors)
    trial.set_user_attr("avg_num_iterations", np.mean(num_iterations_list))
    return avg_error

# Ejecutar Optuna
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=24),
    study_name="exp_pipeline_xgb_gpu_no_scale_5c_2",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=30)

# Mejor conjunto de parámetros

[I 2025-07-13 22:21:50,250] A new study created in RDB with name: exp_pipeline_xgb_gpu_no_scale_5c_2


Train shape: (161112, 659), Test shape: (5538, 659), Fold: 1
[0]	eval-tweedie-nloglik@1.5:8.39951	eval-total_error:1.26868
[100]	eval-tweedie-nloglik@1.5:4.74021	eval-total_error:0.23858


In [14]:
best_params = study.best_params
best_params.update({
    'objective': 'reg:tweedie',
    "device": "cuda",  # Usar GPU
    'tree_method': 'hist',
    'verbosity': 0,
    'sampling_method': 'uniform',  # Aproximación de extra_trees
})

avg_best_iter = int(study.best_trial.user_attrs["avg_num_iterations"])
print(f"🔍 Mejor total_error: {study.best_value:.5f}")
print(f"🏁 Mejor iteración promedio: {avg_best_iter}")
print(f"📊 Mejor conjunto de parámetros: {best_params}")

# Entrenar modelo final
final_df = df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)
X_final = final_df.drop(columns=[col for col in drop_cols if col in final_df.columns])
y_final = final_df["target"]
w_final = np.log1p(final_df["tn"]).clip(lower=1)
dtrain_final = xgb.DMatrix(X_final, label=y_final, weight=w_final, enable_categorical=True)

final_model = xgb.train(
    best_params,
    dtrain=dtrain_final,
    num_boost_round=avg_best_iter,
    verbose_eval=10
)

🔍 Mejor total_error: 0.23663
🏁 Mejor iteración promedio: 144
📊 Mejor conjunto de parámetros: {'tweedie_variance': 1.8680138426687347, 'learning_rate': 0.07295608449546184, 'max_depth': 64, 'lambda': 2.2006729978285176, 'alpha': 3.61056353964058, 'min_child_weight': 74, 'subsample': 0.9982278625445484, 'colsample_bytree': 0.5214428844534258, 'objective': 'reg:tweedie', 'device': 'cuda', 'tree_method': 'hist', 'verbosity': 0, 'sampling_method': 'uniform'}


In [16]:
features = dtrain_final.feature_names
# Reemplazo de inf por nan
kaggle_df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = kaggle_df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        kaggle_df[col] = kaggle_df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = kaggle_df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        kaggle_df.drop(columns=[col], inplace=True)


dfinal_test = xgb.DMatrix(kaggle_df[features], enable_categorical=True)

final_predictions = final_model.predict(dfinal_test)
kaggle_df["predictions"] = final_predictions
# agrupo por product_id y customer_id
final_test_df_grouped = kaggle_df.groupby("product_id").agg({
    "predictions": "sum",
}).reset_index()
final_test_df_grouped = final_test_df_grouped[final_test_df_grouped["product_id"].isin(product_ids)]
submission = final_test_df_grouped[["product_id", "predictions"]].copy()
submission.rename(columns={"predictions": "tn"}, inplace=True)
submission["tn"] = submission["tn"].clip(lower=0)  # Asegurar que no haya valores negativos
submission.to_csv("submission_xgb.csv", index=False)
submission

,product_id,tn
0,20001,1151.046387
1,20002,916.076599
2,20003,748.246582
3,20004,553.062683
4,20005,553.171814
...,...,...
920,21263,0.034547
922,21265,0.054033
923,21266,0.055271
924,21267,0.051337
